# Notebook 06 — Análisis de Datos con Python
## Proyecto Aduana BI · Optativa 4: Inteligencia de Negocios

---

En los notebooks anteriores **construimos** el Data Warehouse: extrajimos datos del Excel,
los limpiamos, los organizamos en dimensiones y los cargamos en una fact table.

Ahora vamos a **usar** ese Data Warehouse directamente desde Python para extraer conclusiones
que van más allá de lo que ofrece Power BI: estadística avanzada, pruebas de hipótesis y machine learning.

### Análisis incluidos en este notebook

| # | Análisis | Técnica principal | Nivel |
|---|----------|------------------|-------|
| 1 | Vista panorámica del comercio exterior | `groupby`, barras, líneas de tiempo | Básico |
| 2 | Principales socios comerciales | Ranking, mapa de calor | Básico |
| 3 | Regla del 80/20 — Pareto | Curva acumulada | Intermedio |
| 4 | ¿Cuánto cuesta importar realmente? | Barras apiladas (CIF) | Intermedio |
| 5 | ¿Quién paga más impuestos? | Tasa tributaria efectiva | Intermedio |
| 6 | ¿Beneficia el MERCOSUR? | Test estadístico Mann-Whitney | Intermedio |
| 7 | El semáforo aduanero | Box plots por canal | Intermedio |
| 8 | Perfiles de países | Clustering K-Means + PCA | Avanzado |
| 9 | Detectar valores inusuales | Método IQR (anomalías) | Avanzado |

---

**Prerequisito:** haber ejecutado los notebooks 01 al 04 (la fact table debe tener datos).


---

## Preparación del entorno

Importamos todas las librerías y configuramos el estilo visual que vamos a usar en todos los gráficos.

Si falta alguna librería, instalarla en la terminal con:
```
pip install duckdb pandas matplotlib seaborn scipy scikit-learn
```


In [ ]:
import os
import warnings

import duckdb                                      # conexión a la base de datos DuckDB
import pandas as pd                               # manipulación de tablas de datos
import matplotlib.pyplot as plt                   # gráficos base
import matplotlib.ticker as mticker               # formato de números en ejes
import seaborn as sns                             # gráficos estadísticos mejorados
from scipy import stats                           # pruebas estadísticas
from sklearn.cluster import KMeans                # algoritmo de agrupamiento K-Means
from sklearn.preprocessing import StandardScaler  # normalización de variables
from sklearn.decomposition import PCA             # reducción de dimensiones
from matplotlib.patches import Patch              # para leyendas manuales

warnings.filterwarnings('ignore')  # ocultamos advertencias menores que no afectan el análisis
%matplotlib inline

# -----------------------------------------------------------
# ESTILO VISUAL UNIFICADO PARA TODOS LOS GRÁFICOS
# Definimos esto una sola vez y aplica a todas las celdas
# -----------------------------------------------------------
plt.rcParams['figure.figsize']    = (13, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid']         = True
plt.rcParams['grid.alpha']        = 0.3
plt.rcParams['font.size']         = 11

# Paleta de colores del proyecto
COLOR_IMPORTACION = '#1A5276'   # azul oscuro  → importaciones
COLOR_EXPORTACION = '#CA6F1E'   # naranja      → exportaciones
COLOR_ALERTA      = '#C0392B'   # rojo         → valores preocupantes
COLOR_OK          = '#1E8449'   # verde        → valores positivos
COLOR_NEUTRO      = '#566573'   # gris         → comparaciones neutrales
PALETA_MULTI      = ['#1A5276','#CA6F1E','#1E8449','#7D3C98','#B03A2E','#117A65','#784212','#1A252F']

print('Librerías cargadas y estilo visual configurado correctamente.')

---

## Conexión a la base de datos

Abrimos el archivo `aduana.duckdb` en modo **solo lectura** (`read_only=True`).
Esta es una buena práctica al hacer análisis: garantizamos que no vamos a modificar
accidentalmente ningún dato del Data Warehouse que construimos.


In [ ]:
# Detectamos la ruta del proyecto automáticamente.
# La carpeta 'notebooks/' está un nivel por debajo de la raíz del proyecto.
carpeta_actual   = os.getcwd()
carpeta_proyecto = (
    os.path.dirname(carpeta_actual)
    if os.path.basename(carpeta_actual) == 'notebooks'
    else carpeta_actual
)
RUTA_DB = os.path.join(carpeta_proyecto, 'db', 'aduana.duckdb')

# read_only=True garantiza que no vamos a modificar el DW al hacer análisis
conexion = duckdb.connect(RUTA_DB, read_only=True)

# Verificamos que la fact table tiene datos antes de continuar
total_registros = conexion.execute('SELECT COUNT(*) FROM dw.fact_aduana_item').fetchone()[0]

print(f'Conectado a: {RUTA_DB}')
print(f'Total de registros en la fact table: {total_registros:,}')

if total_registros == 0:
    print('\nATENCIÓN: La fact table está vacía.')
    print('Ejecutar primero los notebooks 01 al 04 antes de continuar.')

---

## Carga del dataset principal

Unimos la **fact table** con todas las **dimensiones** usando SQL y traemos el resultado
como un DataFrame de pandas llamado `despachos`.

> **¿Por qué traemos todo a pandas y no hacemos cada análisis directo en SQL?**
> Porque para gráficos, clustering y pruebas estadísticas necesitamos los datos en memoria.
> SQL se encarga de la parte relacional (JOINs); pandas y Python hacen el análisis numérico.

> **¿Por qué `LEFT JOIN` y no `INNER JOIN`?**
> El `LEFT JOIN` preserva todos los registros de la fact table aunque alguna dimensión sea NULL.
> Un `INNER JOIN` descartaría esos registros y podríamos perder datos sin darnos cuenta.


In [ ]:
# Construimos la consulta SQL con todos los JOINs necesarios
consulta_principal = '''
    SELECT
        f.despacho_cifrado,
        f.item,

        op.operacion,                        -- IMPORTACION o EXPORTACION

        fec.fecha          AS fecha_ofic,    -- fecha de oficialización
        fec.anio,
        fec.mes_nombre     AS mes,
        fec.mes_numero,
        fec.trimestre,

        ad.aduana,                           -- aduana por donde pasó la mercadería

        po.codigo_pais     AS cod_origen,
        po.descripcion_pais AS pais_origen,
        pd.descripcion_pais AS pais_destino,

        pr.rubro,                            -- categoría general de la mercadería
        pr.desc_capitulo   AS capitulo,
        pr.posicion_ncm,                     -- código arancelario NCM
        pr.mercaderia,

        mt.medio_transporte,
        ca.canal,                            -- V = Verde (automático), R = Rojo (revisión física)
        ac.acuerdo,                          -- MERCOSUR, SIN ACUERDO, etc.

        -- Valores económicos en dólares
        f.fob_dolar,
        f.flete_dolar,
        f.seguro_dolar,
        f.imponible_dolar,

        -- Tributos desglosados
        f.derecho,
        f.isc,
        f.iva,
        f.otros,
        f.total            AS total_tributos,

        -- Peso y cantidad
        f.kilo_neto,
        f.kilo_bruto,
        f.cantidad_estadistica

    FROM dw.fact_aduana_item f
    LEFT JOIN dw.dim_operacion        op  ON f.id_operacion           = op.id_operacion
    LEFT JOIN dw.dim_fecha            fec ON f.id_fecha_oficializacion = fec.id_fecha
    LEFT JOIN dw.dim_aduana           ad  ON f.id_aduana              = ad.id_aduana
    LEFT JOIN dw.dim_pais             po  ON f.id_pais_origen         = po.id_pais
    LEFT JOIN dw.dim_pais             pd  ON f.id_pais_destino        = pd.id_pais
    LEFT JOIN dw.dim_producto         pr  ON f.id_producto            = pr.id_producto
    LEFT JOIN dw.dim_medio_transporte mt  ON f.id_medio_transporte    = mt.id_medio_transporte
    LEFT JOIN dw.dim_canal            ca  ON f.id_canal               = ca.id_canal
    LEFT JOIN dw.dim_acuerdo          ac  ON f.id_acuerdo             = ac.id_acuerdo
    WHERE f.fob_dolar > 0
'''

despachos = conexion.execute(consulta_principal).df()

# Separamos importaciones y exportaciones de antemano porque los usamos en múltiples análisis
importaciones = despachos[despachos['operacion'] == 'IMPORTACION'].copy()
exportaciones = despachos[despachos['operacion'] == 'EXPORTACION'].copy()

print(f'Dataset cargado: {despachos.shape[0]:,} filas x {despachos.shape[1]} columnas')
print(f'  → Importaciones: {len(importaciones):,} ítems')
print(f'  → Exportaciones: {len(exportaciones):,} ítems')
despachos.head(3)

---

## Análisis 1 — Vista panorámica del comercio exterior

Antes de profundizar, necesitamos una vista de alto nivel:
¿cuánto se importó y exportó? ¿en qué meses hay más actividad? ¿cómo evolucionó en el tiempo?

**Técnicas usadas:**
- `groupby()` + `agg()` de pandas para resumir por categorías
- Gráficos de barras y líneas con `matplotlib`
- `mticker.FuncFormatter` para formatear los números del eje (agregar `$`, puntos de miles, etc.)


In [ ]:
# -------------------------------------------------------
# PASO 1: Calcular el resumen por tipo de operación
# -------------------------------------------------------
# groupby + agg es el equivalente Python del GROUP BY de SQL
resumen_operacion = (
    despachos
    .groupby('operacion', dropna=False)
    .agg(
        despachos_unicos = ('despacho_cifrado', 'nunique'),  # cuántos despachos distintos
        total_items      = ('item',             'count'),    # cuántas líneas de ítem
        valor_fob_usd    = ('fob_dolar',         'sum'),     # valor total en dólares
        peso_total_kg    = ('kilo_neto',          'sum'),    # peso total en kg
    )
    .reset_index()
    .sort_values('valor_fob_usd', ascending=False)
)

# Convertimos a millones para que los números sean más legibles al imprimir
resumen_operacion['fob_millones']  = resumen_operacion['valor_fob_usd'] / 1_000_000
resumen_operacion['peso_miles_ton'] = resumen_operacion['peso_total_kg']  / 1_000_000

print('=== RESUMEN POR TIPO DE OPERACIÓN ===\n')
for _, fila in resumen_operacion.iterrows():
    print(
        f"  {fila['operacion']:15s}  "
        f"Despachos: {fila['despachos_unicos']:>5,}  "
        f"Ítems: {fila['total_items']:>6,}  "
        f"FOB: USD {fila['fob_millones']:>7.2f}M  "
        f"Peso: {fila['peso_miles_ton']:>7.2f} mil ton"
    )

In [ ]:
# -------------------------------------------------------
# PASO 2: Gráfico comparativo Importaciones vs Exportaciones
# -------------------------------------------------------
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Panorama del Comercio Exterior de Paraguay', fontsize=14, fontweight='bold')

# Asignamos un color a cada tipo de operación
colores_barra = [
    COLOR_IMPORTACION if op == 'IMPORTACION' else COLOR_EXPORTACION
    for op in resumen_operacion['operacion']
]

# Gráfico izquierdo: cantidad de ítems registrados
ejes[0].bar(resumen_operacion['operacion'], resumen_operacion['total_items'], color=colores_barra)
ejes[0].set_title('Cantidad de ítems registrados')
ejes[0].set_ylabel('Ítems')
ejes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Gráfico derecho: valor FOB total
ejes[1].bar(resumen_operacion['operacion'], resumen_operacion['fob_millones'], color=colores_barra)
ejes[1].set_title('Valor FOB total (millones USD)')
ejes[1].set_ylabel('Millones de USD')
ejes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.1f}M'))

# Leyenda de colores manual
parches = [
    Patch(color=COLOR_IMPORTACION, label='Importación'),
    Patch(color=COLOR_EXPORTACION, label='Exportación'),
]
ejes[1].legend(handles=parches)

plt.tight_layout()
plt.show()

print('\nConcepto: FOB (Free On Board) = valor de la mercadería en el punto de origen,')
print('sin incluir flete ni seguro. Es la medida estándar del comercio internacional.')

In [ ]:
# -------------------------------------------------------
# PASO 3: Evolución mensual del valor FOB
# -------------------------------------------------------
# Calculamos el valor FOB por mes y tipo de operación
fob_mensual = (
    despachos
    .dropna(subset=['mes_numero', 'anio', 'operacion'])
    .groupby(['anio', 'mes_numero', 'mes', 'operacion'])['fob_dolar']
    .sum()
    .reset_index()
    .sort_values(['anio', 'mes_numero'])
)
fob_mensual['fob_millones'] = fob_mensual['fob_dolar'] / 1_000_000

# Creamos una etiqueta legible de período para el eje X
fob_mensual['periodo'] = (
    fob_mensual['anio'].astype(str) + '-' +
    fob_mensual['mes_numero'].astype(str).str.zfill(2)
)

fig, eje = plt.subplots(figsize=(13, 5))

# Graficamos una línea por tipo de operación
for tipo_op, color in [('IMPORTACION', COLOR_IMPORTACION), ('EXPORTACION', COLOR_EXPORTACION)]:
    datos_op = fob_mensual[fob_mensual['operacion'] == tipo_op]
    if not datos_op.empty:
        eje.plot(
            datos_op['periodo'],
            datos_op['fob_millones'],
            marker='o',
            linewidth=2,
            color=color,
            label=tipo_op.capitalize()
        )

eje.set_title('Evolución mensual del valor FOB', fontsize=13)
eje.set_xlabel('Período (año-mes)')
eje.set_ylabel('Millones de USD')
eje.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
eje.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## Análisis 2 — ¿Con quiénes comercia Paraguay?

Identificamos los principales socios comerciales: ¿de qué países vienen la mayoría de
las importaciones? ¿qué aduanas manejan más tráfico y en qué meses?

**Técnicas usadas:**
- Gráfico de barras horizontal (`barh`) — más legible cuando los nombres son largos
- Mapa de calor (`sns.heatmap`) — para ver patrones en una matriz aduana × mes
- `pivot_table()` — equivalente en pandas de una tabla dinámica de Excel


In [ ]:
# Top 10 países de origen de las importaciones por valor FOB
top_paises_origen = (
    importaciones
    .dropna(subset=['pais_origen'])
    .groupby('pais_origen')['fob_dolar']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_paises_origen.columns = ['pais_origen', 'fob_total']
top_paises_origen['fob_millones'] = top_paises_origen['fob_total'] / 1_000_000

fig, eje = plt.subplots(figsize=(10, 6))

# Invertimos el orden para que el mayor quede en la parte superior del gráfico
barras = eje.barh(
    top_paises_origen['pais_origen'][::-1],
    top_paises_origen['fob_millones'][::-1],
    color=COLOR_IMPORTACION
)

# Etiquetas con el valor al costado de cada barra
for barra, valor in zip(barras, top_paises_origen['fob_millones'][::-1]):
    eje.text(
        barra.get_width() + 0.05,
        barra.get_y() + barra.get_height() / 2,
        f'${valor:.2f}M',
        va='center',
        fontsize=9
    )

eje.set_title('Top 10 países de origen de importaciones (por valor FOB)', fontsize=12)
eje.set_xlabel('Valor FOB (millones USD)')
eje.spines['left'].set_visible(False)
eje.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor: ¿qué aduana opera más en cada mes?
datos_heatmap = (
    importaciones
    .dropna(subset=['aduana', 'mes_numero', 'mes'])
    .groupby(['aduana', 'mes_numero', 'mes'])['fob_dolar']
    .sum()
    .reset_index()
    .sort_values('mes_numero')
)

# Orden de meses para las columnas
meses_ordenados = (
    datos_heatmap
    .drop_duplicates('mes_numero')
    .sort_values('mes_numero')['mes']
    .tolist()
)

# pivot_table convierte la tabla larga en una matriz filas=aduana, columnas=mes
tabla_aduana_mes = datos_heatmap.pivot_table(
    index='aduana',
    columns='mes',
    values='fob_dolar',
    aggfunc='sum',
    fill_value=0
)

# Reordenamos las columnas cronológicamente
cols_ordenadas = [m for m in meses_ordenados if m in tabla_aduana_mes.columns]
if cols_ordenadas:
    tabla_aduana_mes = tabla_aduana_mes[cols_ordenadas]

alto_figura = max(4, len(tabla_aduana_mes) * 0.55)
fig, eje = plt.subplots(figsize=(13, alto_figura))

sns.heatmap(
    tabla_aduana_mes / 1_000_000,    # convertir a millones
    ax=eje,
    cmap='Blues',
    annot=True,
    fmt='.1f',
    linewidths=0.5,
    cbar_kws={'label': 'Millones USD'}
)

eje.set_title('Valor FOB de importaciones por aduana y mes (millones USD)', fontsize=12)
eje.set_xlabel('')
eje.set_ylabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nLos colores más oscuros indican mayor volumen de operaciones.')
print('El heatmap es muy útil para detectar estacionalidad y aduanas dominantes.')

---

## Análisis 3 — La Regla del 80/20 (Principio de Pareto)

El economista Vilfredo Pareto observó que el **20% de las causas explican el 80% de los efectos**.
En comercio exterior esto se traduce en:
> *Un pequeño grupo de rubros o productos concentra la mayor parte del valor comerciado.*

Este análisis es fundamental para priorizar recursos en auditorías, controles y negociaciones.

**Técnicas usadas:**
- `cumsum()` — suma acumulada en pandas
- Eje doble (`twinx()`) — para superponer barras y una línea en el mismo gráfico
- Líneas de referencia (`axhline`, `axvline`)


In [ ]:
# Calculamos el valor FOB por rubro y el porcentaje acumulado
fob_por_rubro = (
    importaciones
    .dropna(subset=['rubro'])
    .groupby('rubro')['fob_dolar']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
fob_por_rubro.columns = ['rubro', 'fob_total']

fob_total_general = fob_por_rubro['fob_total'].sum()
fob_por_rubro['porcentaje']           = fob_por_rubro['fob_total'] / fob_total_general * 100
fob_por_rubro['porcentaje_acumulado'] = fob_por_rubro['porcentaje'].cumsum()
fob_por_rubro['fob_millones']         = fob_por_rubro['fob_total'] / 1_000_000

# ¿Cuántos rubros necesitamos para acumular el 80%?
rubros_hasta_80  = (fob_por_rubro['porcentaje_acumulado'] <= 80).sum() + 1
total_rubros     = len(fob_por_rubro)
porcentaje_rubros_80 = rubros_hasta_80 / total_rubros * 100

print(f'Total de rubros diferentes: {total_rubros}')
print(f'Rubros que concentran el 80% del valor FOB: {rubros_hasta_80}')
print(f'Eso representa el {porcentaje_rubros_80:.1f}% del total de rubros')
print(f'\nTop 5 rubros:')
print(fob_por_rubro[['rubro','fob_millones','porcentaje','porcentaje_acumulado']].head(5).to_string(index=False))

# --- Gráfico de Pareto ---
fig, eje_barras = plt.subplots(figsize=(14, 6))
eje_acum = eje_barras.twinx()    # segundo eje Y a la derecha para la curva acumulada

# Barras del valor FOB por rubro
eje_barras.bar(
    range(len(fob_por_rubro)),
    fob_por_rubro['fob_millones'],
    color=COLOR_IMPORTACION,
    alpha=0.75,
    label='FOB por rubro'
)

# Curva de porcentaje acumulado
eje_acum.plot(
    range(len(fob_por_rubro)),
    fob_por_rubro['porcentaje_acumulado'],
    color=COLOR_ALERTA,
    linewidth=2.5,
    label='% acumulado'
)

# Líneas de referencia al 80%
eje_acum.axhline(y=80, color='gray', linestyle='--', linewidth=1.2, label='Línea 80%')
if rubros_hasta_80 <= len(fob_por_rubro):
    eje_barras.axvline(x=rubros_hasta_80 - 0.5, color='gray', linestyle='--', linewidth=1.2)

eje_barras.set_title('Análisis de Pareto — Valor FOB por rubro de importación', fontsize=13)
eje_barras.set_ylabel('Valor FOB (millones USD)')
eje_acum.set_ylabel('Porcentaje acumulado (%)')
eje_acum.set_ylim(0, 112)

# Solo mostramos etiquetas para los primeros rubros (evitamos saturar el eje)
max_etiq = min(20, len(fob_por_rubro))
nombres_truncados = fob_por_rubro['rubro'].str[:25]
eje_barras.set_xticks(range(max_etiq))
eje_barras.set_xticklabels(nombres_truncados[:max_etiq], rotation=45, ha='right', fontsize=8)

lineas_b, etiq_b = eje_barras.get_legend_handles_labels()
lineas_a, etiq_a = eje_acum.get_legend_handles_labels()
eje_barras.legend(lineas_b + lineas_a, etiq_b + etiq_a, loc='center right')

plt.tight_layout()
plt.show()

---

## Análisis 4 — ¿Cuánto cuesta realmente traer la mercadería? (Valor CIF)

Cuando hablamos del costo real de una importación, el precio FOB es solo una parte.
El valor que se usa como base para calcular tributos es el **CIF**:

```
CIF = FOB + Flete + Seguro
      (Cost, Insurance and Freight)
```

El flete y el seguro varían mucho según el medio de transporte:
traer algo por avión es mucho más caro que por barco, pero más rápido.

**Técnicas usadas:**
- Barras apiladas (`bottom=`) — para mostrar la composición interna de un total
- `fillna(0)` — para reemplazar valores nulos por cero (mercadería sin flete declarado)


In [ ]:
# Calculamos el CIF para cada ítem
importaciones_cif = importaciones.copy()
importaciones_cif['cif_dolar'] = (
    importaciones_cif['fob_dolar'] +
    importaciones_cif['flete_dolar'].fillna(0) +
    importaciones_cif['seguro_dolar'].fillna(0)
)

# Agrupamos por medio de transporte y sumamos los componentes
cif_por_transporte = (
    importaciones_cif
    .dropna(subset=['medio_transporte'])
    .groupby('medio_transporte')
    .agg(
        total_fob    = ('fob_dolar',    'sum'),
        total_flete  = ('flete_dolar',  'sum'),
        total_seguro = ('seguro_dolar', 'sum'),
        total_cif    = ('cif_dolar',    'sum'),
    )
    .reset_index()
    .sort_values('total_cif', ascending=False)
)

# Calculamos qué porcentaje del CIF corresponde a flete y seguro
cif_por_transporte['pct_flete']  = cif_por_transporte['total_flete']  / cif_por_transporte['total_cif'] * 100
cif_por_transporte['pct_seguro'] = cif_por_transporte['total_seguro'] / cif_por_transporte['total_cif'] * 100
cif_por_transporte['pct_fob']    = cif_por_transporte['total_fob']    / cif_por_transporte['total_cif'] * 100

print('Composición del CIF por medio de transporte:')
print(f"{'Transporte':<20} {'% FOB':>8} {'% Flete':>9} {'% Seguro':>10}")
print('-' * 50)
for _, fila in cif_por_transporte.iterrows():
    print(f"  {fila['medio_transporte']:<18} {fila['pct_fob']:>7.1f}% {fila['pct_flete']:>8.1f}% {fila['pct_seguro']:>9.1f}%")

# --- Gráfico de barras apiladas ---
fig, eje = plt.subplots(figsize=(10, 5))

transportes = cif_por_transporte['medio_transporte']
posiciones  = range(len(transportes))

# Cada barra se dibuja sobre la anterior usando el parámetro 'bottom'
fob_m = cif_por_transporte['total_fob']    / 1_000_000
fl_m  = cif_por_transporte['total_flete']  / 1_000_000
sg_m  = cif_por_transporte['total_seguro'] / 1_000_000

eje.bar(posiciones, fob_m,               color='#1A5276', label='FOB')
eje.bar(posiciones, fl_m,  bottom=fob_m, color='#2980B9', label='Flete')
eje.bar(posiciones, sg_m,  bottom=fob_m + fl_m, color='#85C1E9', label='Seguro')

eje.set_xticks(posiciones)
eje.set_xticklabels(transportes, rotation=15)
eje.set_title('Composición del costo CIF por medio de transporte', fontsize=13)
eje.set_ylabel('Millones de USD')
eje.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.1f}M'))
eje.legend()
plt.tight_layout()
plt.show()

print('\nEl transporte aéreo suele tener mayor proporción de flete/seguro que el acuático.')
print('Eso hace que el CIF sea proporcionalmente más alto para productos aéreos.')

---

## Análisis 5 — ¿Quién paga más impuestos? (Carga tributaria por rubro)

No todos los productos pagan la misma tasa de impuestos. Algunos rubros tienen protección
arancelaria alta (bienes que el Estado quiere proteger o desincentivar), mientras que otros
entran casi sin impuestos.

Calculamos la **tasa tributaria efectiva** de cada rubro:

```
Tasa efectiva (%) = (Derecho + ISC + IVA + Otros) / Valor Imponible × 100
```

> **¿Por qué sobre el total del rubro y no el promedio de tasas individuales?**
> Porque promediar porcentajes sin ponderar por monto es engañoso.
> Un ítem de $1 con 50% de tasa no debería pesar igual que uno de $1.000.000.


In [ ]:
# Solo consideramos importaciones con valor imponible declarado
imp_con_valor = importaciones[importaciones['imponible_dolar'] > 0].copy()

# Sumamos todos los tributos en una sola columna
imp_con_valor['tributos_totales'] = (
    imp_con_valor['derecho'].fillna(0) +
    imp_con_valor['isc'].fillna(0) +
    imp_con_valor['iva'].fillna(0) +
    imp_con_valor['otros'].fillna(0)
)

# Agrupamos por rubro y calculamos la tasa efectiva sobre los totales
carga_por_rubro = (
    imp_con_valor
    .dropna(subset=['rubro'])
    .groupby('rubro')
    .agg(
        total_imponible = ('imponible_dolar',  'sum'),
        total_tributos  = ('tributos_totales',  'sum'),
        cantidad_items  = ('item',              'count'),
    )
    .reset_index()
)

carga_por_rubro['tasa_efectiva_pct'] = (
    carga_por_rubro['total_tributos'] / carga_por_rubro['total_imponible'] * 100
)

# Top 15 rubros con mayor carga tributaria efectiva
top_carga = carga_por_rubro.sort_values('tasa_efectiva_pct', ascending=False).head(15)

fig, eje = plt.subplots(figsize=(10, 7))

# Coloreamos en rojo los rubros con tasa > 30% como llamada de atención
colores_carga = [
    COLOR_ALERTA if t > 30 else COLOR_IMPORTACION
    for t in top_carga['tasa_efectiva_pct'][::-1]
]

barras = eje.barh(
    top_carga['rubro'].str[:40][::-1],
    top_carga['tasa_efectiva_pct'][::-1],
    color=colores_carga
)

for barra, tasa in zip(barras, top_carga['tasa_efectiva_pct'][::-1]):
    eje.text(
        barra.get_width() + 0.2,
        barra.get_y() + barra.get_height() / 2,
        f'{tasa:.1f}%',
        va='center',
        fontsize=9
    )

eje.set_title('Top 15 rubros con mayor carga tributaria efectiva en importaciones', fontsize=12)
eje.set_xlabel('Tasa efectiva = tributos / valor imponible (%)')
eje.spines['left'].set_visible(False)
eje.tick_params(axis='y', length=0)

leyenda_colores = [
    Patch(color=COLOR_ALERTA,      label='Tasa > 30% (alta protección)'),
    Patch(color=COLOR_IMPORTACION, label='Tasa ≤ 30%'),
]
eje.legend(handles=leyenda_colores)

plt.tight_layout()
plt.show()

---

## Análisis 6 — ¿Beneficia el MERCOSUR a Paraguay? (Test estadístico)

El MERCOSUR es un acuerdo de libre comercio entre Argentina, Brasil, Paraguay y Uruguay.
La teoría dice que importar desde países del MERCOSUR debería tener tasas de derecho más bajas.

Pero para afirmar esto con rigor, no basta con comparar promedios — necesitamos un
**test estadístico** que nos diga si la diferencia observada es real o podría ser
simplemente producto del azar.

Usamos el **Test de Mann-Whitney U** porque:
- Las tasas de derecho no tienen distribución normal (hay muchos ceros y valores extremos)
- Mann-Whitney es un test no paramétrico: no asume ninguna distribución específica
- Compara si los valores de un grupo tienden a ser mayores/menores que los del otro

**Hipótesis:**
- H₀ (nula): No hay diferencia en las tasas entre MERCOSUR y SIN ACUERDO
- H₁ (alternativa): Hay diferencia significativa
- Si el valor p < 0.05 → rechazamos H₀ → la diferencia es estadísticamente significativa


In [ ]:
# Calculamos la tasa de derecho efectiva por ítem
imp_mercosur = importaciones[
    (importaciones['imponible_dolar'] > 0) &
    (importaciones['acuerdo'].notna())
].copy()

imp_mercosur['tasa_derecho_pct'] = (
    imp_mercosur['derecho'].fillna(0) / imp_mercosur['imponible_dolar'] * 100
)

# Resumen descriptivo por tipo de acuerdo
resumen_acuerdo = (
    imp_mercosur
    .groupby('acuerdo')['tasa_derecho_pct']
    .agg(
        cantidad   = 'count',
        media      = 'mean',
        mediana    = 'median',
        maximo     = 'max'
    )
    .reset_index()
    .sort_values('cantidad', ascending=False)
)
print('=== RESUMEN POR ACUERDO COMERCIAL ===\n')
print(resumen_acuerdo.to_string(index=False))

# Aplicamos el test de Mann-Whitney
grupo_mercosur    = imp_mercosur[imp_mercosur['acuerdo'] == 'MERCOSUR']['tasa_derecho_pct'].dropna()
grupo_sin_acuerdo = imp_mercosur[imp_mercosur['acuerdo'] == 'SIN ACUERDO']['tasa_derecho_pct'].dropna()

if len(grupo_mercosur) > 1 and len(grupo_sin_acuerdo) > 1:
    estadistico_u, valor_p = stats.mannwhitneyu(
        grupo_mercosur,
        grupo_sin_acuerdo,
        alternative='two-sided'
    )
    print(f'\n=== TEST DE MANN-WHITNEY U ===')
    print(f'MERCOSUR     n={len(grupo_mercosur):,}   mediana={grupo_mercosur.median():.2f}%')
    print(f'SIN ACUERDO  n={len(grupo_sin_acuerdo):,}   mediana={grupo_sin_acuerdo.median():.2f}%')
    print(f'Estadístico U: {estadistico_u:,.0f}')
    print(f'Valor p:       {valor_p:.6f}')
    if valor_p < 0.05:
        print('\n→ Diferencia SIGNIFICATIVA (p < 0.05)')
        print('  El acuerdo MERCOSUR sí tiene efecto estadísticamente demostrable')
        print('  sobre la tasa de derecho de importación.')
    else:
        print('\n→ Diferencia NO significativa (p >= 0.05)')
        print('  No hay suficiente evidencia para afirmar que el MERCOSUR reduce las tasas.')
else:
    print('No hay suficientes datos en ambos grupos para realizar el test.')

In [ ]:
# Visualizamos la distribución de tasas con box plots
acuerdos_a_comparar = ['MERCOSUR', 'SIN ACUERDO']
datos_boxplot = imp_mercosur[
    imp_mercosur['acuerdo'].isin(acuerdos_a_comparar) &
    (imp_mercosur['tasa_derecho_pct'] < 100)   # excluimos valores extremos para escalar mejor
].copy()

fig, eje = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=datos_boxplot,
    x='acuerdo',
    y='tasa_derecho_pct',
    palette={'MERCOSUR': COLOR_OK, 'SIN ACUERDO': COLOR_ALERTA},
    width=0.45,
    ax=eje,
    order=acuerdos_a_comparar
)

eje.set_title('Distribución de la tasa de derecho por acuerdo comercial', fontsize=12)
eje.set_xlabel('')
eje.set_ylabel('Tasa de derecho efectiva (%)')

plt.tight_layout()
plt.show()

print('\nLectura del box plot:')
print('  Línea central = mediana (valor del 50% de los datos)')
print('  Caja          = rango intercuartílico Q1-Q3 (50% central de los datos)')
print('  Bigotes       = hasta 1.5 × IQR fuera de la caja')
print('  Puntos        = valores atípicos (outliers)')

---

## Análisis 7 — El semáforo aduanero (canal de despacho)

En aduana, cada despacho recibe un **canal** que determina el nivel de control:

| Canal | Nombre | Significado |
|-------|--------|-------------|
| **V** | Verde | Liberación automática sin revisión |
| **R** | Rojo | Revisión física de la mercadería |

Los sistemas de gestión de riesgo aduanero determinan este canal en base a perfiles.
La pregunta analítica es: **¿los despachos de canal Rojo difieren significativamente
de los de canal Verde en términos de valor, peso y tributos?**

Si la respuesta es sí, el sistema está funcionando correctamente: está eligiendo para
revisión física los despachos de mayor riesgo o valor.


In [ ]:
# Filtramos solo los canales V y R, ignoramos valores nulos o inusuales
despachos_canal = despachos[despachos['canal'].isin(['V', 'R'])].copy()
despachos_canal['nombre_canal'] = despachos_canal['canal'].map({
    'V': 'Verde (automático)',
    'R': 'Rojo (físico)'
})

# Estadísticas descriptivas por canal
resumen_canal = (
    despachos_canal
    .groupby('nombre_canal')
    .agg(
        cantidad_items      = ('item',            'count'),
        fob_mediana         = ('fob_dolar',        'median'),
        fob_promedio        = ('fob_dolar',        'mean'),
        peso_mediano_kg     = ('kilo_neto',        'median'),
        tributos_mediana    = ('total_tributos',   'median'),
    )
    .reset_index()
)
print('=== COMPARACIÓN POR CANAL DE DESPACHO ===\n')
print(resumen_canal.to_string(index=False))

# Gráfico: box plots de 3 variables por canal
fig, ejes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Comparación de despachos por canal (Verde vs Rojo)', fontsize=13)

variables_canal = [
    ('fob_dolar',      'Valor FOB (USD)'),
    ('kilo_neto',      'Peso neto (kg)'),
    ('total_tributos', 'Total tributos (USD)'),
]

paleta_canal = {
    'Verde (automático)': COLOR_OK,
    'Rojo (físico)':      COLOR_ALERTA
}

for eje, (variable, etiqueta) in zip(ejes, variables_canal):
    # Filtramos valores positivos (necesario para escala logarítmica)
    datos_var = despachos_canal[despachos_canal[variable] > 0][['nombre_canal', variable]]

    sns.boxplot(
        data=datos_var,
        x='nombre_canal',
        y=variable,
        palette=paleta_canal,
        ax=eje,
        width=0.45,
        showfliers=False,   # ocultamos outliers extremos para ver mejor la caja
        order=['Verde (automático)', 'Rojo (físico)']
    )

    eje.set_title(etiqueta)
    eje.set_xlabel('')
    eje.set_ylabel('')

    # Escala logarítmica para variables muy asimétricas
    if datos_var[variable].max() / (datos_var[variable].median() + 1) > 50:
        eje.set_yscale('log')
        eje.set_ylabel('Escala logarítmica')

    eje.tick_params(axis='x', labelrotation=15)

plt.tight_layout()
plt.show()

print('\nNota: showfliers=False oculta los outliers extremos para ver mejor la forma de la distribución.')
print('La escala logarítmica permite comparar distribuciones muy asimétricas (muchos valores pequeños, pocos muy grandes).')

---

## Análisis 8 — Perfiles de países (Clustering K-Means)

Hasta ahora vimos los países por ranking de valor FOB.
Pero ¿existen **grupos de países con perfiles similares** más allá del volumen?

El **clustering** (agrupamiento) es una técnica de aprendizaje automático **no supervisado**:
le damos los datos y el algoritmo encuentra grupos por sí solo, sin que le digamos de antemano
cuáles son esos grupos.

Usamos **K-Means**: divide los datos en K grupos minimizando la distancia interna.

Después, para poder graficar los grupos en 2D, usamos **PCA** (Análisis de Componentes Principales):
reduce las 5 variables originales a solo 2 dimensiones que capturan la mayor varianza posible.

**Variables del perfil de cada país:**
- FOB promedio por ítem (¿qué tan cara es su mercadería?)
- Peso neto promedio (¿volumen o valor?)
- Flete promedio (¿qué tan lejos está?)
- Diversidad de rubros (¿proveedor especializado o generalista?)
- % de ítems con MERCOSUR (¿parte del bloque regional?)


In [ ]:
# Construimos el perfil de cada país de origen
MIN_ITEMS_POR_PAIS = 5    # excluimos países con muy pocos registros

perfil_paises = (
    importaciones
    .dropna(subset=['pais_origen'])
    .groupby('pais_origen')
    .agg(
        total_items         = ('item',            'count'),
        fob_promedio        = ('fob_dolar',        'mean'),
        peso_promedio_kg    = ('kilo_neto',        'mean'),
        flete_promedio      = ('flete_dolar',      'mean'),
        diversidad_rubros   = ('rubro',            'nunique'),
        pct_mercosur        = ('acuerdo',          lambda x: (x == 'MERCOSUR').mean() * 100),
    )
    .reset_index()
)

perfil_paises = perfil_paises[perfil_paises['total_items'] >= MIN_ITEMS_POR_PAIS].copy()
print(f'Países con al menos {MIN_ITEMS_POR_PAIS} ítems: {len(perfil_paises)}')
print(f'\nPrimeros 5 perfiles:')
perfil_paises.head()

In [ ]:
# Preparamos las variables para el clustering
variables_para_clustering = ['fob_promedio', 'peso_promedio_kg', 'flete_promedio',
                              'diversidad_rubros', 'pct_mercosur']

X = perfil_paises[variables_para_clustering].fillna(0).values

# StandardScaler: lleva todas las variables a media=0 y desviación=1
# Sin normalizar, el FOB en miles de USD dominaría sobre diversidad_rubros (que es un número pequeño)
normalizador  = StandardScaler()
X_normalizado = normalizador.fit_transform(X)

# Método del codo: entrenamos K-Means para k=2 a k=8 y graficamos la inercia
# La inercia mide qué tan compactos son los clusters (menor = mejor, pero crece con k)
# El 'codo' en la curva es el k óptimo
rango_k  = range(2, min(9, len(perfil_paises)))
inercias = []

for k in rango_k:
    modelo_kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo_kmeans.fit(X_normalizado)
    inercias.append(modelo_kmeans.inertia_)

fig, eje = plt.subplots(figsize=(8, 4))
eje.plot(list(rango_k), inercias, marker='o', color=COLOR_IMPORTACION, linewidth=2.5)
eje.set_title('Método del codo para elegir el número óptimo de clusters', fontsize=12)
eje.set_xlabel('Número de clusters (k)')
eje.set_ylabel('Inercia')
eje.set_xticks(list(rango_k))
plt.tight_layout()
plt.show()

print('Elegir el k donde la curva deja de bajar bruscamente (el "codo").')
print('Modificar K_OPTIMO en la siguiente celda según lo que observe aquí.')

In [ ]:
# Aplicar K-Means con el k elegido
K_OPTIMO = 3    # ← ajustar según el codo del gráfico anterior

kmeans_final = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
perfil_paises['cluster'] = kmeans_final.fit_predict(X_normalizado)

# PCA: reduce las 5 variables a 2 dimensiones para poder visualizarlas
# Los componentes principales son combinaciones lineales que capturan la máxima varianza
pca_modelo    = PCA(n_components=2, random_state=42)
componentes   = pca_modelo.fit_transform(X_normalizado)
perfil_paises['componente_1'] = componentes[:, 0]
perfil_paises['componente_2'] = componentes[:, 1]

varianza_explicada = pca_modelo.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada por los 2 componentes: {varianza_explicada:.1f}%')
print('(Cuanto más cerca del 100%, más fiel es la visualización 2D a los datos reales)')

# Resumen de cada cluster
resumen_clusters = (
    perfil_paises
    .groupby('cluster')
    .agg(
        cantidad_paises   = ('pais_origen',     'count'),
        fob_promedio      = ('fob_promedio',     'mean'),
        flete_promedio    = ('flete_promedio',   'mean'),
        pct_mercosur_prom = ('pct_mercosur',     'mean'),
        diversidad_prom   = ('diversidad_rubros','mean'),
    )
    .reset_index()
)
print('\nPerfil de cada cluster:')
print(resumen_clusters.to_string(index=False))

# Diagrama de dispersión en el espacio PCA
fig, eje = plt.subplots(figsize=(11, 7))

for cluster_id in range(K_OPTIMO):
    mask = perfil_paises['cluster'] == cluster_id
    datos_cluster = perfil_paises[mask]
    eje.scatter(
        datos_cluster['componente_1'],
        datos_cluster['componente_2'],
        s=datos_cluster['total_items'] * 2,   # tamaño del punto = volumen del país
        color=PALETA_MULTI[cluster_id],
        alpha=0.75,
        edgecolors='white',
        linewidths=0.5,
        label=f'Cluster {cluster_id}'
    )

# Etiquetamos los países de mayor volumen
top_etiquetas = perfil_paises.nlargest(8, 'total_items')
for _, fila in top_etiquetas.iterrows():
    eje.annotate(
        fila['pais_origen'][:20],
        (fila['componente_1'], fila['componente_2']),
        xytext=(6, 4),
        textcoords='offset points',
        fontsize=8,
        color='#2C3E50'
    )

eje.set_title(
    f'Agrupamiento de países por perfil comercial ({K_OPTIMO} clusters)\n'
    f'Visualizado con PCA — varianza explicada: {varianza_explicada:.1f}%',
    fontsize=12
)
eje.set_xlabel('Componente Principal 1')
eje.set_ylabel('Componente Principal 2')
eje.legend()

plt.tight_layout()
plt.show()

print('\nEl tamaño del punto es proporcional al volumen de ítems importados desde ese país.')
print('Países cercanos en el gráfico tienen perfiles comerciales similares.')

---

## Análisis 9 — Detección de valores inusuales (Anomalías con IQR)

En comercio exterior, la **subvaloración** es el principal mecanismo de evasión fiscal:
declarar un valor FOB menor al real para pagar menos impuestos.

Una señal de alerta es un precio por kilogramo (`FOB / kg`) inusualmente bajo para su rubro.

**Método IQR (Rango Intercuartílico):**
```
IQR = Q3 − Q1
Límite inferior = Q1 − 1.5 × IQR
Límite superior = Q3 + 1.5 × IQR
```
Cualquier valor fuera de esos límites es considerado **atípico** dentro de su rubro.

> Este mismo método es la base de los sistemas de **gestión de riesgo aduanero** reales.
> La aduana compara el precio declarado contra el rango estadístico de su base histórica.


In [ ]:
# Calculamos el precio por kilogramo para cada ítem
datos_precio = importaciones[
    (importaciones['kilo_neto'] > 0) &
    (importaciones['fob_dolar'] > 0) &
    (importaciones['rubro'].notna())
].copy()

datos_precio['precio_por_kg'] = datos_precio['fob_dolar'] / datos_precio['kilo_neto']

# Función que aplica el método IQR a una serie y retorna una máscara booleana
def detectar_anomalias_iqr(serie):
    q1       = serie.quantile(0.25)
    q3       = serie.quantile(0.75)
    rango_iqr = q3 - q1
    limite_inferior = q1 - 1.5 * rango_iqr
    limite_superior = q3 + 1.5 * rango_iqr
    return (serie < limite_inferior) | (serie > limite_superior)

# Aplicamos la detección DENTRO de cada rubro (no en el dataset global)
# Esto es clave: el precio de un kilo de algodón no se compara con el de un kilo de perfume
datos_precio['es_anomalia'] = (
    datos_precio
    .groupby('rubro')['precio_por_kg']
    .transform(detectar_anomalias_iqr)
)

total_analizados = len(datos_precio)
total_anomalias  = datos_precio['es_anomalia'].sum()
pct_anomalias    = total_anomalias / total_analizados * 100

print(f'Total de ítems analizados:         {total_analizados:,}')
print(f'Ítems con precio inusual (anomalía): {total_anomalias:,} ({pct_anomalias:.1f}%)')

# ¿Qué rubros tienen más anomalías?
anomalias_por_rubro = (
    datos_precio
    .groupby('rubro')
    .agg(
        total_items      = ('item',           'count'),
        total_anomalias  = ('es_anomalia',    'sum'),
        precio_mediano   = ('precio_por_kg',  'median'),
    )
    .reset_index()
)
anomalias_por_rubro['pct_anomalias'] = (
    anomalias_por_rubro['total_anomalias'] / anomalias_por_rubro['total_items'] * 100
)

top_anomalias = anomalias_por_rubro.sort_values('pct_anomalias', ascending=False).head(12)

fig, eje = plt.subplots(figsize=(11, 6))

barras = eje.barh(
    top_anomalias['rubro'].str[:40][::-1],
    top_anomalias['pct_anomalias'][::-1],
    color=[
        COLOR_ALERTA if p > 20 else COLOR_IMPORTACION
        for p in top_anomalias['pct_anomalias'][::-1]
    ]
)

for barra, pct in zip(barras, top_anomalias['pct_anomalias'][::-1]):
    eje.text(
        barra.get_width() + 0.3,
        barra.get_y() + barra.get_height() / 2,
        f'{pct:.1f}%',
        va='center',
        fontsize=9
    )

eje.set_title('Rubros con mayor porcentaje de precios inusuales (método IQR)', fontsize=12)
eje.set_xlabel('% de ítems con precio FOB/kg inusual')
eje.spines['left'].set_visible(False)
eje.tick_params(axis='y', length=0)

leyenda_anomalia = [
    Patch(color=COLOR_ALERTA,      label='> 20% de ítems anómalos (riesgo alto)'),
    Patch(color=COLOR_IMPORTACION, label='≤ 20% de ítems anómalos'),
]
eje.legend(handles=leyenda_anomalia)

plt.tight_layout()
plt.show()

print('\nImportante: una anomalía estadística NO implica necesariamente fraude.')
print('Puede deberse a mercadería de calidad excepcional, lotes pequeños, o errores de registro.')
print('Es una señal de alerta que justifica una revisión más profunda, no una conclusión definitiva.')

In [ ]:
# Mostramos una muestra de los despachos con precio inusualmente bajo
# (precio por kg muy por debajo del Q1 de su rubro → posible subvaloración)
anomalias_bajas = datos_precio[
    datos_precio['es_anomalia'] & (datos_precio['precio_por_kg'] <
    datos_precio.groupby('rubro')['precio_por_kg'].transform('quantile', 0.25))
].sort_values('fob_dolar', ascending=False)

columnas_a_mostrar = ['despacho_cifrado', 'rubro', 'mercaderia', 'pais_origen',
                      'fob_dolar', 'kilo_neto', 'precio_por_kg', 'aduana']

print(f'Ítems con precio FOB/kg inusualmente BAJO: {len(anomalias_bajas):,}')
print('(Ordenados por valor FOB — los de mayor valor son los más relevantes para control)\n')
anomalias_bajas[columnas_a_mostrar].head(10)

---

## Conclusión y próximos pasos

En este notebook aplicamos **9 técnicas de análisis** directamente sobre el Data Warehouse
que construimos en los notebooks anteriores:

| Técnica | ¿Para qué sirve en la práctica? |
|---------|----------------------------------|
| `groupby` + visualización | Entender la estructura general del negocio |
| Análisis de Pareto | Priorizar dónde enfocar recursos de control |
| Composición CIF | Entender el costo real de importar |
| Tasa tributaria efectiva | Comparar la carga fiscal entre sectores |
| Test Mann-Whitney | Validar estadísticamente si un acuerdo tiene efecto |
| Box plots por canal | Evaluar si el sistema de riesgo está bien calibrado |
| K-Means + PCA | Descubrir perfiles ocultos en los datos |
| Método IQR | Detectar señales de alerta para auditoría |

---

### ¿Qué sigue?

Posibles extensiones para trabajos finales:

1. **Clasificador de canal** — Entrenar un `RandomForestClassifier` para predecir si
   un despacho irá al canal V o R antes de que llegue a la aduana.

2. **Forecasting** — Usar `statsmodels` o `Prophet` para proyectar el volumen de
   importaciones del próximo trimestre.

3. **Red de proveedores** — Modelar las relaciones país → rubro → aduana como un grafo
   con `networkx` para detectar patrones de concentración.

4. **Integración completa** — Conectar este análisis Python con un reporte automático
   en Word/PDF usando `python-docx` o `reportlab`.
